# 08 — Configurations

Configurations let one part file contain several design variants
(think: 1/4", 1/2", 3/4" sizes). Each configuration can suppress
features or override parameter values independently.

## What you'll do
1. Start with a box that has a fillet
2. Add two configurations (one locked, one editable)
3. Switch between them
4. Suppress the fillet in one configuration only

**Prereq:** open a fresh empty part in Alibre.

## Setup: box with a fillet

In [ ]:
from alibrex import (
    CurrentPart,
    connect_to_running_alibre,
    ADDirectionType,
    ADPartFeatureEndCondition,
)

part = CurrentPart()

xy = part.DesignPlanes.Item(0)
sk = part.Sketches.AddSketch(None, xy, "Base")
figs = sk.Figures
figs.AddLine(0.0, 0.0, 4.0, 0.0)
figs.AddLine(4.0, 0.0, 4.0, 3.0)
figs.AddLine(4.0, 3.0, 0.0, 3.0)
figs.AddLine(0.0, 3.0, 0.0, 0.0)
part.Features.AddExtrudedBoss(
    sk, 1.5, ADPartFeatureEndCondition.AD_TO_DEPTH,
    None, None, 0.0,
    ADDirectionType.AD_ALONG_NORMAL, None, None, False,
    None, False,
    "Box", "Depth", "",
)

## Add a fillet on every edge

`NewObjectCollector` builds the set of edges to fillet. It lives on
the root, hence the one-line connect call.

In [ ]:
root = connect_to_running_alibre()
body = part.Bodies.Item(0)
edges = root.NewObjectCollector()
for i in range(body.Edges.Count):
    edges.Add(body.Edges.Item(i))

fillet = part.Features.AddConstantRadiusFilletFeature(
    edges,
    0.2,                # radius (cm)
    True,               # tangent-propagate
    "",                 # parameter name (none)
    "AllFillets",       # feature name
)
fillet.Name

## How many configurations exist by default?

In [ ]:
configs = part.Configurations
configs.Count        # 1 — the default config Alibre creates

## Add two new configurations

In [ ]:
foo = configs.AddConfiguration("Foo", False)   # editable
bar = configs.AddConfiguration("Bar", True)    # locked
configs.Count

## Switch between them

In [ ]:
part.ActiveConfiguration = foo
part.ActiveConfiguration.Name

In [ ]:
part.ActiveConfiguration = bar
part.ActiveConfiguration.Name

## Suppress the fillet in `Foo` only

Switch to Foo, suppress the feature, switch to Bar, see it's still
unsuppressed.

In [ ]:
part.ActiveConfiguration = foo
fillet.Suppressed = True
part.RegenerateAll()
fillet.Suppressed

In [ ]:
part.ActiveConfiguration = bar
part.RegenerateAll()
fillet.Suppressed

## List configurations with their state

In [ ]:
for i in range(configs.Count):
    c = configs.Item(i)
    print(f"  {c.Name:20s} id={c.ID}")